# Analytics — Consultas Gold

Consultas analíticas sobre a camada Gold: receita, popularidade, ranking, participações e lucro por produtora.

In [0]:
from pyspark.sql.functions import col, sum, desc, count, rank, max, add_months, lit, current_date
from pyspark.sql.window import Window

# Leituras das tabelas Gold
df_movies = spark.read.table("cinedata_lakehouse.gold.dim_movies")
df_fact = spark.read.table("cinedata_lakehouse.gold.fact_movies_performance")
df_genres = spark.read.table("cinedata_lakehouse.gold.dim_genres")
df_bridge_genre = spark.read.table("cinedata_lakehouse.gold.bridge_movie_genre")
df_people = spark.read.table("cinedata_lakehouse.gold.dim_people")
df_bridge_person = spark.read.table("cinedata_lakehouse.gold.bridge_movie_person")
df_companies = spark.read.table("cinedata_lakehouse.gold.dim_companies")
df_bridge_company = spark.read.table("cinedata_lakehouse.gold.bridge_movie_company")


# 1. Qual é a receita total (em R$) somada de todos os filmes da base?

display(
    df_fact.select(sum("receita_brl").alias("receita_total_brl_acumulada"))
)


# 2. Quais são os 5 filmes com maior popularidade?

display(
    df_movies.join(df_fact, on="sk_movie_id", how="inner")
    .filter(col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(col("popularidade").desc_nulls_last())
    .limit(5)
)

# 3. Quantos filmes cada gênero possui? (Do maior para o menor)

display(
    df_bridge_genre.join(df_genres, on="sk_genre_id", how="inner")
    .groupBy("nome_genero")
    .agg(count("sk_movie_id").alias("quantidade_filmes"))
    .orderBy(desc("quantidade_filmes"))
)


# 4. Para os 10 filmes de maior receita: título, receita USD, receita BRL e RANK()


window_receita = Window.partitionBy(lit(1)).orderBy(desc("receita_usd"))

display(
    df_movies.join(df_fact, on="sk_movie_id", how="inner")
    .filter(col("receita_usd").isNotNull())
    .orderBy(desc("receita_usd"))
    .limit(10)
    .withColumn("ranking_receita", rank().over(window_receita))
    .select("titulo", "receita_usd", "receita_brl", "ranking_receita")
)


data_maxima_row = (
    df_movies
    .filter(col("status_filme") == "Lançado")
    .filter(col("data_lancamento") <= current_date()) # Proteção Crítica contra o bug de 2029
    .select(max("data_lancamento").alias("data_max"))
    .collect()[0]
)
data_maxima = data_maxima_row["data_max"]


# 5. Qual ator teve a maior quantidade de participações nos últimos 2 anos?

display(
    df_movies
    .filter(col("data_lancamento").isNotNull())
    .filter(col("data_lancamento") >= add_months(lit(data_maxima), -24))
    .join(df_bridge_person, on="sk_movie_id", how="inner")
    .join(df_people, on="sk_person_id", how="inner")
    .filter(col("tipo_pessoa") == "Ator")
    .groupBy("nome_pessoa")
    .agg(count("sk_movie_id").alias("qtd_participacoes"))
    .orderBy(desc("qtd_participacoes"))
    .limit(1)
)


# 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?

display(
    df_movies
    .filter(col("data_lancamento").isNotNull())
    .filter(col("data_lancamento") >= add_months(lit(data_maxima), -60))
    .join(df_fact, on="sk_movie_id", how="inner")
    .filter(col("lucro_usd").isNotNull())
    .join(df_bridge_company, on="sk_movie_id", how="inner")
    .join(df_companies, on="sk_company_id", how="inner")
    .groupBy("nome_produtora")
    .agg(sum("lucro_usd").alias("lucro_total_usd_acumulado"))
    .orderBy(col("lucro_total_usd_acumulado").desc_nulls_last())
    .limit(1)
)